# Phase X scratch notebook
Copy this file to `notebooks/<phase>_<topic>.ipynb` (for example `E_regression.ipynb`) and work there.
Save work-in-progress files to `data/scratch/<phase>/`. When it works, move finished cells into your section of `fantasy_football_rankings.ipynb` (see "Avoiding conflicts" in `AGENTS.md`).

Before the first run, run `bash scripts/verify.sh` once from the repo root so `data/processed/` exists.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
PROCESSED_DIR = ROOT / "data" / "processed"

# Same values as the main notebook's config cell. Change them there, not here.
SEED = 363
POSITIONS = ["QB", "RB", "WR", "TE"]
TRAIN_SEASONS = list(range(2020, 2025))
TEST_SEASON = 2025
TARGET = "fantasy_points_ppr"
CV_VALID_SEASONS = [2022, 2023, 2024]
TOP_N = 12

## Shared data
`features` has one row per player-week (2020-2025) with the target. `FEATURE_FAMILIES` lists the only columns that may be used as model inputs. `train` drops 2025, which is only for the final test (Phase H).

In [2]:
features = pd.read_parquet(PROCESSED_DIR / "features.parquet")
FEATURE_FAMILIES = json.loads((PROCESSED_DIR / "feature_families.json").read_text())
ALL_FEATURES = [c for cols in FEATURE_FAMILIES.values() for c in cols]

train = features[features["season"].isin(TRAIN_SEASONS)].reset_index(drop=True)
print(f"train: {train.shape}, features: {len(ALL_FEATURES)}")

train: (18402, 55), features: 48


## Shared evaluation (copied exactly from main notebook section 5.6; a test checks they match)

In [3]:
def cv_folds(data):
    for season in CV_VALID_SEASONS:
        train = data.index[data["season"].between(TRAIN_SEASONS[0], season - 1)]
        valid = data.index[data["season"] == season]
        yield season, train, valid


def top_n_hit_rate(week):
    predicted_top = set(week.nlargest(TOP_N, "pred").index)
    actual_top = set(week.nlargest(TOP_N, TARGET).index)
    return len(predicted_top & actual_top) / TOP_N


def evaluate(data, pred):
    d = data.assign(pred=np.asarray(pred))
    groups = d.groupby(["season", "week", "position"])
    return {
        "rmse": float(np.sqrt(((d["pred"] - d[TARGET]) ** 2).mean())),
        "mae": float((d["pred"] - d[TARGET]).abs().mean()),
        "spearman": float(groups.apply(lambda w: w["pred"].corr(w[TARGET], method="spearman")).mean()),
        "top12_hit_rate": float(groups.apply(top_n_hit_rate).mean()),
    }


def cross_validate(model, data, columns):
    """Fit on each expanding-window fold; return mean metrics and the out-of-fold predictions."""
    from sklearn.base import clone
    preds = pd.Series(np.nan, index=data.index)
    for season, train, valid in cv_folds(data):
        fitted = clone(model).fit(data.loc[train, columns], data.loc[train, TARGET])
        preds.loc[valid] = fitted.predict(data.loc[valid, columns])
    scored = preds.notna()
    return evaluate(data[scored], preds[scored]), preds

## Smoke test: should print Spearman 0.5146 (the Ridge result from section 5.8)

In [4]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
scores, oof_preds = cross_validate(ridge, train, ALL_FEATURES)
{k: round(v, 4) for k, v in scores.items()}

{'rmse': 6.809, 'mae': 5.1939, 'spearman': 0.5146, 'top12_hit_rate': 0.4896}